# 공공문서 RAG 검색 평가 실습 - 전처리 포함

PDF 로드, 메타데이터 정규화, 문서 분할부터 검색 평가까지 전체 과정을 실습한다.

### 실습 목표

- PDF를 페이지 단위로 불러오고 평가에 필요한 메타데이터를 구성한다.
- 문서를 청크로 나누어 벡터 저장소를 구성한다.
- File Hit@k, Page Hit@k, MRR을 계산한다.
- Similarity Search와 MMR의 검색 성능을 비교한다.
- 실패 문항의 검색 결과를 확인하고 원인을 분석한다.

### 제공 데이터

- 검색 대상 문서: `data/public/*.pdf`
- 평가 데이터: `data/public/public_paragraph_golden_set.json`

Golden set의 `question`은 검색 질의, `target_answer`는 기준 답변, `target_file_name`과 `target_page_no`는 정답 위치를 나타낸다.

In [1]:
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
load_dotenv(override=True)

DATA_DIR = Path("data/public")
GOLDEN_SET_PATH = DATA_DIR / "public_paragraph_golden_set.json"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

C:\Users\puh\AppData\Local\Temp\ipykernel_19492\1014861403.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Golden set 확인

평가 문항 수와 필드 구성을 확인한다.

In [2]:
eval_cases = json.loads(GOLDEN_SET_PATH.read_text(encoding="utf-8"))

print(f"평가 문항: {len(eval_cases)}개")
print(json.dumps(eval_cases[:5], ensure_ascii=False, indent=2))

평가 문항: 35개
[
  {
    "domain": "public",
    "question": "2023년 국가재정운용계획 수립 절차 중, 분야별·지역별 예산협의회 및 간담회 개최 시기와 국회 제출 시기는 언제인가요?",
    "target_answer": "2023년 국가재정운용계획 수립 절차에 따르면, 분야별·지역별 예산협의회 및 간담회는 2023년 4월부터 8월까지 개최되고, 국회 제출은 2023년 9월 1일에 이루어집니다.",
    "target_file_name": "2023_2027 국가재정운용계획 주요내용.pdf",
    "target_page_no": 3,
    "context_type": "paragraph"
  },
  {
    "domain": "public",
    "question": "불요불급한 자산을 처분하여 회수하는 자금은 어떻게 활용되는지, 그리고 공공기관의 재무건전성 관리 강화를 위해 '22~'26년 재정건전화계획 수정계획을 수립하는 것 외에 어떤 방법이 적용되는지 설명해주세요.",
    "target_answer": "불요불급한 자산을 처분하여 회수된 자금은 재무건전성 제고 등에 활용됩니다. 또한, 재무건전성 관리 강화를 위해 대내외 여건 변화를 반영하여 재무위험기관의 '22~'26년 재정건전화계획의 수정계획을 수립하고, 공공기관의 혁신계획과 재정건전화계획 등을 반영한 중장기 재무관리계획을 수립하여 국회에 제출합니다.",
    "target_file_name": "2023_2027 국가재정운용계획 주요내용.pdf",
    "target_page_no": 22,
    "context_type": "paragraph"
  },
  {
    "domain": "public",
    "question": "북한의 위협에 대응하기 위해 강화해야하는 한국형 3축체계와는 무엇이며, 이를 통해 향상시키려는 국방 분야의 세부적인 전투능력은 무엇인가요?",
    "target_answer": "한국형 3축체계는 북

### PDF 로드와 메타데이터 정규화

모든 PDF를 페이지 단위로 불러오는 코드는 제공한다. 로드된 문서의 메타데이터를 다음 조건에 맞게 정리하세요.

- `source`: 전체 경로가 아닌 파일명
- `page_no`: 1부터 시작하는 페이지 번호

In [5]:
page_documents = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    pages = PyPDFLoader(pdf_path).load()

    for page in pages:
        # TODO: 조건에 맞게 메타데이터를 정리하세요.
        page.metadata["source"] = pdf_path.name
        page.metadata["page_no"] = page.metadata["page"] + 1

    page_documents.extend(pages)

print(f"로드한 PDF: {len(list(DATA_DIR.glob('*.pdf')))}개")
print(f"로드한 페이지: {len(page_documents)}개")
print(page_documents[0].metadata)

로드한 PDF: 11개
로드한 페이지: 236개
{'producer': 'Hancom PDF 1.3.0.546', 'creator': 'Hwp 2020 11.0.0.8362', 'creationdate': '2024-04-11T11:32:01+09:00', 'moddate': '2024-04-11T11:32:01+09:00', 'pdfversion': '1.4', 'source': '(240411보도자료) 재정동향 4월호.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'page_no': 1}


### 문서 분할

페이지 정보를 유지하면서 `chunk_size=700`, `chunk_overlap=100`으로 문서를 분할하세요.

In [8]:
# TODO: 조건에 맞게 documents를 구성하세요.
documents = []

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, 
    chunk_overlap=70
    )
documents = splitter.split_documents(page_documents)

print(f"생성한 청크: {len(documents)}개")

생성한 청크: 393개


### 벡터 저장소와 검색기 구성

동일한 문서로 Similarity Search와 MMR을 구성한다. 이 실습에서는 별도 저장 경로 없이 `Chroma.from_documents()`로 벡터 저장소를 만들고, 검색기 딕셔너리의 키는 `Similarity`와 `MMR`을 사용한다.

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

# TODO: vectorstore와 retrievers를 구성하세요.
vectorstore = None
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
)

In [11]:
retrievers = {}

retrievers["Similarity"] = vectorstore.as_retriever(search_kwargs={"k": 5})
retrievers["MMR"] = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k" : 20, "lambda_mult" : 0.5})

print(retrievers)

{'Similarity': VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B7BE0A3ED0>, search_kwargs={'k': 5}), 'MMR': VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B7BE0A3ED0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 20, 'lambda_mult': 0.5})}


### 평가 지표 구현

File Hit은 정답 파일의 검색 여부를, Page Hit은 정답 파일과 페이지가 함께 일치하는지를 확인한다. MRR은 최초로 등장한 정답 페이지 순위의 역수이다.

검색 결과 문서 `doc`과 golden set 문항 `case`는 다음 값으로 비교한다.

```python
파일 일치: doc.metadata["source"] == case["target_file_name"]
페이지 일치: doc.metadata["page_no"] == case["target_page_no"]
정답 페이지: 파일 일치 and 페이지 일치
```

예를 들어 정답 페이지가 검색 결과의 세 번째에 처음 등장하면 RR은 `1 / 3`이다. 상위 k개에 정답이 없으면 Hit은 0이고, 전체 검색 결과에 정답 페이지가 없으면 RR은 0.0이다.

In [21]:
def file_hit_at_k(docs, case, k):
    # TODO
    for doc in docs[:k]:
        if doc.metadata["source"] == case["target_file_name"]:
            return 1
    return 0


def page_hit_at_k(docs, case, k):
    for doc in docs[:k]:
        if doc.metadata["source"] == case["target_file_name"] and doc.metadata["page_no"] == case["target_page_no"]:
            return 1
    return 0


def reciprocal_rank(docs, case):
    # TODO
    for rank, doc in enumerate(docs, start=1):
        if doc.metadata["source"] == case["target_file_name"] and doc.metadata["page_no"] == case["target_page_no"]:
            return 1 / rank
    return 0.0

### 전체 평가

모든 질문을 검색하고 전략별 지표와 실행 시간을 기록한다. latency는 `retriever.invoke()` 한 번에 걸린 시간을 밀리초 단위로 측정한다. 실패 원인을 확인할 수 있도록 검색 결과의 순위, 파일명, 페이지 번호도 함께 저장한다.

In [16]:
import time

def evaluate_retriever(name, retriever, cases):
    rows = []
    # TODO: 평가 결과를 rows에 저장하세요.

    for case in cases:
        start = time.perf_counter()
        docs = retriever.invoke(case["question"])
        latency_ms = (time.perf_counter() - start) * 1000

        rows.append({
            "name" : name,
            "question" : case["question"],
            "file_hit_1" : file_hit_at_k(docs, case, 1),
            "page_hit_1" : page_hit_at_k(docs, case, 1),
            "page_hit_3" : page_hit_at_k(docs, case, 3),
            "page_hit_5" : page_hit_at_k(docs, case, 5),
            "reciprocal_rank" : reciprocal_rank(docs, case),
            "latency_ms" : latency_ms,
            "target_file_name" : case["target_file_name"],
            "target_page_no" : case["target_page_no"],
            "retrieved" : [
                {
                    "rank" : rank,
                    "source" : doc.metadata["source"],
                    "page_no" : doc.metadata["page_no"],
                }
                for rank, doc in enumerate(docs, start=1)
            ]
        })

    return rows

In [22]:
# TODO: 모든 검색 전략을 평가하세요.
evaluation_results = []

for name, retriever in retrievers.items():
    rows = evaluate_retriever(name, retriever, eval_cases)
    evaluation_results.extend(rows)

print(json.dumps(evaluation_results[:5], ensure_ascii=False, indent=2))

[
  {
    "name": "Similarity",
    "question": "2023년 국가재정운용계획 수립 절차 중, 분야별·지역별 예산협의회 및 간담회 개최 시기와 국회 제출 시기는 언제인가요?",
    "file_hit_1": 1,
    "page_hit_1": 1,
    "page_hit_3": 1,
    "page_hit_5": 1,
    "reciprocal_rank": 1.0,
    "latency_ms": 196.4937997981906,
    "target_file_name": "2023_2027 국가재정운용계획 주요내용.pdf",
    "target_page_no": 3,
    "retrieved": [
      {
        "rank": 1,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 3
      },
      {
        "rank": 2,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 1
      },
      {
        "rank": 3,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 2
      },
      {
        "rank": 4,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 7
      },
      {
        "rank": 5,
        "source": "2024년+국립대학+육성사업+기본계획.pdf",
        "page_no": 21
      }
    ]
  },
  {
    "name": "Similarity",
    "question": "불요불급한 자산을 처분하여 회수하는 자금은 어떻게 활용되는지, 그리고 공공기관의 재무건

### 결과 비교

검색 전략별 평균 점수를 비교한다. 전략마다 문항 수 `count`와 `file_hit_1`, `page_hit_1`, `page_hit_3`, `page_hit_5`, `reciprocal_rank`, `latency_ms`의 평균을 계산한다. 여러 문항의 `reciprocal_rank` 평균이 MRR이다.

In [23]:
# TODO: 검색 전략별 결과를 요약하세요.
summary = []
def average(rows, key):
    return sum(row[key] for row in rows) / len(rows) if rows else 0.0

for retriever in retrievers:
    rows = [row for row in evaluation_results if row["name"] == retriever]

    summary.append({
        "name" : retriever,
        "file_hit_1" : average(rows, "file_hit_1"),
        "page_hit_1" : average(rows, "page_hit_1"),
        "page_hit_3" : average(rows, "page_hit_3"),
        "page_hit_5" : average(rows, "page_hit_5"),
        "reciprocal_rank" : average(rows, "reciprocal_rank"),
        "latency_ms" : average(rows, "latency_ms"),
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

[
  {
    "name": "Similarity",
    "file_hit_1": 0.8857142857142857,
    "page_hit_1": 0.6285714285714286,
    "page_hit_3": 0.7142857142857143,
    "page_hit_5": 0.8,
    "reciprocal_rank": 0.6819047619047618,
    "latency_ms": 153.78752278962307
  },
  {
    "name": "MMR",
    "file_hit_1": 0.8857142857142857,
    "page_hit_1": 0.6285714285714286,
    "page_hit_3": 0.7142857142857143,
    "page_hit_5": 0.7142857142857143,
    "reciprocal_rank": 0.6666666666666666,
    "latency_ms": 149.970299936831
  }
]


### 실패 문항 분석

전체 평가 결과에서 `page_hit_5 == 0`인 행을 확인한다. 각 실패 행에 저장한 `retrieved`를 정답 파일·페이지와 비교하여 같은 파일의 다른 페이지가 검색되었는지, 다른 파일만 검색되었는지 분석한다.

In [24]:
# TODO: 실패 문항을 추출하세요.
failures = [row for row in evaluation_results if row["page_hit_5"] == 0]

print(json.dumps(failures, ensure_ascii=False, indent=2))

[
  {
    "name": "Similarity",
    "question": "사업비 정산에 대한 절차 중 국고지원금 집행 시점과 사업종료 후 발생이자에 대한 처리 방안에 대해서 설명하시오.",
    "file_hit_1": 0,
    "page_hit_1": 0,
    "page_hit_3": 0,
    "page_hit_5": 0,
    "reciprocal_rank": 0.0,
    "latency_ms": 163.80579955875874,
    "target_file_name": "2024년+국립대학+육성사업+기본계획.pdf",
    "target_page_no": 19,
    "retrieved": [
      {
        "rank": 1,
        "source": "240319 알기쉬운 청년농신보 책자(최종본)★_배포용.pdf",
        "page_no": 16
      },
      {
        "rank": 2,
        "source": "2024년 행정안전부 업무계획.pdf",
        "page_no": 9
      },
      {
        "rank": 3,
        "source": "240319 알기쉬운 청년농신보 책자(최종본)★_배포용.pdf",
        "page_no": 12
      },
      {
        "rank": 4,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 18
      },
      {
        "rank": 5,
        "source": "2023_2027 국가재정운용계획 주요내용.pdf",
        "page_no": 21
      }
    ]
  },
  {
    "name": "Similarity",
    "question": "고향사랑기부제 2.0에서는 어떤 방식으로 기부한도 상향 및 기부방식을 개선

### 결과 정리

다음 내용을 작성한다.

- 가장 성능이 좋은 검색 전략과 판단 근거
- File Hit과 Page Hit 결과가 다른 이유
- 실패 문항에서 확인한 검색 결과의 특징
- 성능을 개선하기 위해 변경해 볼 설정